# The DS inner loop with mbt

This notebook is the data scientist's lab bench for the showcase's churn model.
The philosophy: the notebook is where you EXPLORE and ANALYZE; the model itself always lives in reviewed YAML ("the model config IS the model").
Nothing trained from ad-hoc notebook calls can reach the registry - training goes through `mbt build`, which pins data snapshots, seeds, and gates, so a teammate or the scheduler reproduces your result without your notebook.

Run this inside the showcase's JupyterLab (`make up`, then http://localhost:8899, open `project/notebooks/`).
Background reading: `docs/tutorial.md` (the two-persona story) and `DESIGN.md` in the showcase root.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
from pyarrow import fs

# The kernel starts in notebooks/; every mbt command runs from the project root.
if not Path("mbt_project.yml").exists() and Path("/workspace/project/mbt_project.yml").exists():
    os.chdir("/workspace/project")
assert Path("mbt_project.yml").exists(), "run this notebook from the showcase JupyterLab"

# The showcase pins its anchor to the seeded cohorts. In production the
# scheduler passes the Airflow logical date here.
ANCHOR = "2026-06-30T00:00:00Z"
print("project root:", Path.cwd())

## 1. Explore the lake table

The whole data model is ONE table, `s3://mbt-lake/churn_panel` (browse it at http://localhost:8388/buckets/): one row per customer per month-start `inference_date`, with the `is_active` population flag, every feature, and the `is_churn` label.
It is wide - most of its columns are ones no model uses - so read only the columns you are asking about.

In [ ]:
lake = fs.S3FileSystem(
    endpoint_override="seaweedfs:8333",
    scheme="http",
    access_key=os.environ["AWS_ACCESS_KEY_ID"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
panel = ds.dataset("mbt-lake/churn_panel", filesystem=lake, format="parquet")
print(f"{panel.count_rows()} rows x {len(panel.schema)} columns")
cohorts = panel.to_table(columns=["inference_date", "is_active", "is_churn"]).to_pandas()
cohorts.groupby("inference_date").agg(
    rows=("is_active", "size"),
    active=("is_active", "sum"),
    labelled=("is_churn", "count"),
    churn_rate=("is_churn", "mean"),
).round(3)

Two things the summary shows are what the specs are built around:

- `is_active` is the population: churned customers keep their rows, marked inactive, so the dataset and the scoring node both filter on it.
- The newest cohort (2026-06-01) has no labels yet - its outcome window is still open. It is the batch the champion scores, and its labels land in the same table a month later (`make outcomes`).

## 2. The model is YAML

You do not define the training set in this notebook - you review the declaration.
The dataset spec below is everything mbt decides about the rows: the one table it reads, the population filter, the label and its horizon, sampling, and the exact train/test cohort boundaries.
The model specs decide the columns: 16 named features out of the table's full width.
Editing any of it flips the model's config hash, so CI retrains exactly what changed.

In [ ]:
print(Path("datasets/churn_training.yml").read_text())

## 3. The fast inner loop: build on the dev target

`mbt build` compiles a pinned manifest, materializes the dataset with Spark over the s3a lake, trains, evaluates gates, and writes machine-readable results.
Exit codes mean something: 0 trained and passed, 2 means a gate or check said no (feedback, not breakage), 1 is a real error.

In [ ]:
!mbt build --target dev --select +churn_baseline_xgb --anchor 2026-06-30T00:00:00Z

## 4. Analyze what the run produced

Everything mbt writes is a file the notebook can read: metrics and gates in `target/run_results.json`, and the EXACT frame the model saw under `target/datasets/`.
This is where notebook strengths matter - slice the training set, inspect importances, question the gates.

In [ ]:
results = json.loads(Path("target/run_results.json").read_text())
model = next(r for r in results["results"] if r["unique_id"].endswith(".churn_baseline_xgb"))
print("metrics:", {k: round(v, 4) for k, v in model["metrics"].items()})
print("gates:", model["gates"])
pd.Series(model.get("feature_importance") or {}).sort_values(ascending=False).head(10)

In [ ]:
newest = max(
    Path("target/datasets/churn_training").glob("*/train.parquet"),
    key=lambda p: p.stat().st_mtime,
)
train = pd.read_parquet(
    newest, columns=["inference_date", "login_days_30d", "txn_cnt_30d", "is_churn"]
)
print(f"train split: {train.shape[0]} rows")
train.groupby("inference_date").mean().round(3)

## 5. Experiment on a sample without touching the contract

For quick what-ifs, hash sampling gives you a coherent slice: `sample_fraction: 0.25` keeps every cohort of a quarter of the customers, and smaller fractions are subsets of larger ones.
A `--vars` override is a one-off: nothing committed changes.

In [ ]:
subprocess.run(
    [
        "mbt",
        "build",
        "--target",
        "dev",
        "--select",
        "+churn_baseline_xgb",
        "--anchor",
        ANCHOR,
        "--vars",
        "sample_fraction: 0.25",
    ],
    check=True,
)
sampled = json.loads(Path("target/run_results.json").read_text())
sampled_model = next(
    r for r in sampled["results"] if r["unique_id"].endswith(".churn_baseline_xgb")
)
pd.DataFrame({"full": model["metrics"], "25% sample": sampled_model["metrics"]}).round(4)

## 6. Ship it: the notebook ends where the PR begins

When the spec looks right, everything you decided is already sitting in files a reviewer can read: the dataset and model YAML.
Commit them on a branch and open a PR (in the showcase: clone `http://localhost:3305/mbt-showcase/churn` after `make ci`).
From there the platform takes over, and none of it needs this notebook:

- the PR check state-diffs against the published baseline, retrains ONLY what you changed on a throwaway registry, and posts the metrics and gate verdicts as a comment;
- the merge build retrains what changed and bakes a digest-pinned deployable unit; a reviewed edit to `promotions.yml` flips the production alias - zero redeploy;
- the `mbt_score` DAG scores the newest cohort with the run-time champion, and `mbt_monitor` evaluates realized metrics once the cohort's labels land in the same table.

Your experiment became infrastructure the moment you committed the YAML.